In [0]:
for s in ["raw", "clean", "analytics"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{s}")

display(spark.sql("SHOW SCHEMAS IN workspace"))

databaseName
analytics
clean
default
information_schema
raw
taxi_weather


In [0]:
import requests, json, time
from pyspark.sql import functions as F

URL = "https://archive-api.open-meteo.com/v1/archive"
PARAMS = {
    "latitude": 40.71, "longitude": -74.01,
    "start_date": "2016-01-01", "end_date": "2016-02-29",
    "hourly": "temperature_2m,precipitation",
    "timezone": "America/New_York",
}
FALLBACK = "/Volumes/workspace/taxi_weather/raw/weather_nyc_2016.json"

def fetch_api(retries=3):
    for i in range(retries):
        try:
            r = requests.get(URL, params=PARAMS, timeout=30)
            if r.status_code == 200:
                return r.text
            print(f"Attempt {i+1}: status {r.status_code}")
            if r.status_code != 429 and r.status_code < 500:
                return None  # client error, retrying won't help
        except requests.RequestException as e:
            print(f"Attempt {i+1}: {type(e).__name__}")
        time.sleep(5 * (i + 1))
    return None

payload = fetch_api()
source = "open-meteo-api"
if payload is None:
    with open(FALLBACK) as f:
        payload = f.read()
    source = "file-fallback"

# basic validation before writing
hours = len(json.loads(payload)["hourly"]["time"])
assert hours == 1440, f"Unexpected hours: {hours}"

raw_weather = (
    spark.createDataFrame([(payload, json.dumps(PARAMS), source)],
                          ["payload", "request_params", "_source"])
         .withColumn("_ingested_at", F.current_timestamp())
)
raw_weather.write.mode("overwrite").saveAsTable("workspace.raw.weather_raw")

display(spark.table("workspace.raw.weather_raw")
        .select("_source", "_ingested_at", F.length("payload").alias("payload_chars")))

_source,_ingested_at,payload_chars
open-meteo-api,2026-11-23T17:40:25.559Z,41489
